In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_SILENT"] = "true"


In [ ]:
os.environ["WANDB_DISABLED"] = "true"


In [ ]:
# Install needed packages (Path B needs transformers/datasets)
!pip install -q imbalanced-learn transformers datasets evaluate sentencepiece nlpaug

import os, re, random, joblib
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline

print("Libraries ready.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 16.2 MB/s eta 0:00:00
Libraries ready.


In [ ]:
CSV = "Combined Data.csv"  # change if needed
df = pd.read_csv(CSV, on_bad_lines='skip', engine='python')
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
df = df.rename(columns={'statement':'text','Statement':'text','status':'label','Status':'label'})
df = df[['text','label']].dropna().drop_duplicates().reset_index(drop=True)
print("Rows:", len(df))
def clean_text(s):
    s = str(s).lower()
    s = re.sub(r"http\S+|www\S+|https\S+", " ", s)
    s = re.sub(r"[@#]", " ", s)
    s = re.sub(r"[^a-z0-9\s']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s
df['clean_text'] = df['text'].apply(clean_text)
print(df['label'].value_counts())


Rows: 11068
label
Normal        6183
Suicidal      2099
Depression    2074
Anxiety        712
Name: count, dtype: int64


In [ ]:
labels = sorted(df['label'].unique().tolist())
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}
print("Labels:", labels)

df['label_id'] = df['label'].map(label2id)

from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df[['clean_text','label','label_id']],
                                     test_size=0.2, random_state=42, stratify=df['label_id'])

from datasets import Dataset
hf_train = Dataset.from_pandas(train_df.rename(columns={'clean_text':'text'})).remove_columns('__index_level_0__')
hf_test  = Dataset.from_pandas(test_df.rename(columns={'clean_text':'text'})).remove_columns('__index_level_0__')

def map_label(example):
    example['label'] = int(label2id[example['label']])
    return example

hf_train = hf_train.map(map_label)
hf_test  = hf_test.map(map_label)
print("HF train/test sizes:", len(hf_train), len(hf_test))


Labels: ['Anxiety', 'Depression', 'Normal', 'Suicidal']


Map:   0%|          | 0/8854 [00:00<?, ? examples/s]

Map:   0%|          | 0/2214 [00:00<?, ? examples/s]

HF train/test sizes: 8854 2214


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

hf_train = hf_train.map(tokenize, batched=True)
hf_test  = hf_test.map(tokenize, batched=True)

hf_train = hf_train.rename_column("label", "labels")
hf_test  = hf_test.rename_column("label", "labels")

hf_train.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
hf_test.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
print("Tokenization done.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/8854 [00:00<?, ? examples/s]

Map:   0%|          | 0/2214 [00:00<?, ? examples/s]

Tokenization done.


In [ ]:
# Robust Trainer creation + train cell (paste & run)
import traceback
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np, evaluate

# sanity checks
needed = ['hf_train','hf_test','tokenizer','labels','label2id','id2label']
missing = [v for v in needed if v not in globals()]
if missing:
    raise RuntimeError("Missing variables in runtime: " + ", ".join(missing) +
                       ". Re-run cells that create hf_train/hf_test/tokenizer/label mappings.")

num_labels = len(labels)
print("num_labels =", num_labels)

# load model (classification head will be new)
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

metric = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, gold = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=gold, average="macro")

# Try modern TrainingArguments first; otherwise fall back to minimal
from packaging import version
import transformers
print("transformers version:", transformers.__version__)

trainer = None
try:
    print("Trying to create TrainingArguments with modern kwargs...")
    training_args = TrainingArguments(
        output_dir="distilbert-mh",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        logging_steps=50,
        load_best_model_at_end=True,
        fp16=True
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=hf_train,
        eval_dataset=hf_test,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )
    print("Trainer created with modern TrainingArguments.")
except TypeError as e:
    print("Modern TrainingArguments failed with TypeError (falling back).")
    print("TypeError:", e)
    # print stack for debug
    traceback.print_exc()
    # fallback minimal args (compatible with old transformers)
    training_args = TrainingArguments(
        output_dir="distilbert-mh",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        logging_steps=50
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=hf_train,
        eval_dataset=hf_test,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )
    print("Trainer created with fallback TrainingArguments (no evaluation_strategy).")

# If trainer created, start training
if trainer is None:
    raise RuntimeError("Trainer could not be created. Check earlier errors.")
else:
    print("Starting trainer.train() — this will run fine with either arg set.")
    trainer.train()
    trainer.save_model("distilbert_mh_model")
    tokenizer.save_pretrained("distilbert_mh_model")
    print("Training finished and model saved to distilbert_mh_model/")


num_labels = 7


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Traceback (most recent call last):
  File "/tmp/ipython-input-3272209845.py", line 38, in <cell line: 0>
    training_args = TrainingArguments(
                    ^^^^^^^^^^^^^^^^^^
TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


transformers version: 4.57.3
Trying to create TrainingArguments with modern kwargs...
Modern TrainingArguments failed with TypeError (falling back).
TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'


/tmp/ipython-input-3272209845.py:71: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Trainer created with fallback TrainingArguments (no evaluation_strategy).
Starting trainer.train() — this will run fine with either arg set.


Step,Training Loss
50,1.428300
100,1.188400
150,0.979500
200,0.899300
250,0.916000
300,0.838000
350,0.822800
400,0.871300
450,0.785100
500,0.733300


Training finished and model saved to distilbert_mh_model/


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# y_true = labels from pred.label_ids
# y_pred = np.argmax(logits, axis=-1)

target_names = [id2label[i] for i in range(len(id2label))]

print(classification_report(
    y_true,
    y_pred,
    target_names=target_names,
    digits=4
))

cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix:\n", cm)


NameError: name 'y_true' is not defined

In [ ]:
# load (if not in memory) and minimal predictor that prints only the emotion
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch, numpy as np

tokenizer = AutoTokenizer.from_pretrained("distilbert_mh_model")
model = AutoModelForSequenceClassification.from_pretrained("distilbert_mh_model")
model.eval()

# mapping id -> label (ensure id2label exists; else rebuild)
# if id2label not in memory:
# labels = sorted(df['label'].unique().tolist()); id2label = {i:l for i,l in enumerate(labels)}

def predict_distil_label_only(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    pred_id = int(np.argmax(probs))
    return id2label[pred_id]

s = input().strip()
if s:
    print(predict_distil_label_only(s))


My mind keeps racing and I feel tense about everything
Stress
